In [4]:
def generate_test_data_fixed(
    n_genes: int = 1000,
    n_samples_per_group: int = 5,
    n_groups: int = 2,
    n_batches: int = 2,
    de_ratio: float = 0.3,  # 差异基因比例
    seed: int = 42
) -> tuple:
    """
    修正版：支持多组之间都有差异
    """
    np.random.seed(seed)
    
    # 1. 生成基因名称和长度
    gene_names = [f"gene_{i+1}" for i in range(n_genes)]
    gene_lengths = np.random.lognormal(mean=8, sigma=0.8, size=n_genes)
    gene_lengths = np.round(gene_lengths).astype(int)
    gene_lengths = np.clip(gene_lengths, 500, 50000)
    length_dict = {gene: length for gene, length in zip(gene_names, gene_lengths)}
    
    # 2. 生成样本元数据
    group_names = [f"group_{i+1}" for i in range(n_groups)]
    batch_names = [f"batch_{i+1}" for i in range(n_batches)]
    
    sample_names = []
    group_assignments = []
    batch_assignments = []
    sex_assignments = []
    
    for g_idx, group in enumerate(group_names):
        for s_idx in range(n_samples_per_group):
            sample_name = f"{group}_sample_{s_idx+1}"
            sample_names.append(sample_name)
            group_assignments.append(group)
            batch = np.random.choice(batch_names)
            batch_assignments.append(batch)
            sex = np.random.choice(['M', 'F'])
            sex_assignments.append(sex)
    
    metadata = pd.DataFrame({
        'condition': group_assignments,
        'batch': batch_assignments,
        'sex': sex_assignments
    }, index=sample_names)
    
    # 3. 生成计数矩阵
    counts_matrix = np.zeros((n_genes, len(sample_names)))
    
    # 基础表达水平
    base_expr = np.random.lognormal(mean=5, sigma=1.5, size=n_genes)
    base_expr = base_expr / base_expr.mean() * 1000
    
    # 为每个基因随机选择一个"基准组"（作为参照）
    # 其他组相对于基准组可能有差异
    for g_idx in range(n_genes):
        # 随机决定这个基因是否差异表达
        is_de = np.random.random() < de_ratio
        
        for s_idx, sample in enumerate(sample_names):
            group = group_assignments[s_idx]
            batch = batch_assignments[s_idx]
            
            # 基础表达量
            expr_mean = base_expr[g_idx]
            
            # 如果这个基因是差异表达基因，且不是基准组
            if is_de:
                # 随机选择一个组作为"处理组"（可以是任意组）
                # 这里简化：让 group_2 和 group_3 相对于 group_1 有差异
                if group == group_names[1]:  # group_2
                    fold_change = np.random.choice([-1, 1]) * np.random.uniform(1, 3.3)
                    expr_mean = expr_mean * (2 ** fold_change)
                elif n_groups > 2 and group == group_names[2]:  # group_3
                    fold_change = np.random.choice([-1, 1]) * np.random.uniform(1, 2.5)
                    expr_mean = expr_mean * (2 ** fold_change)
            
            # 批次效应
            if batch == batch_names[1]:
                expr_mean = expr_mean * np.random.uniform(0.8, 1.2)
            
            # 生成计数
            dispersion = 0.2
            if expr_mean > 0:
                overdispersion = np.random.gamma(shape=max(0.1, expr_mean/dispersion), 
                                                scale=dispersion)
                count = np.random.poisson(overdispersion)
            else:
                count = 0
            
            counts_matrix[g_idx, s_idx] = max(0, int(count))
    
    # 转换为DataFrame
    counts = pd.DataFrame(
        counts_matrix,
        index=gene_names,
        columns=sample_names
    )
    counts = counts.astype(int)
    
    # 添加低表达基因
    low_expr_genes = np.random.choice(n_genes, size=int(n_genes*0.1), replace=False)
    for g_idx in low_expr_genes:
        zero_samples = np.random.choice(len(sample_names), 
                                       size=int(len(sample_names)*0.8), 
                                       replace=False)
        for s_idx in zero_samples:
            counts.iloc[g_idx, s_idx] = np.random.poisson(2)
    
    return counts, metadata, length_dict

In [ ]:

from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os 
from sklearn.decomposition import PCA
from scipy.stats import false_discovery_control


class CountData:
    """
    RNA-seq 计数数据的核心容器类
    存储counts矩阵,metadata和基因长度,提供基本的数据操作接口
    
    counts:pd.DataFrame
        index:基因名/特征ID
        columns:样本名
    metadata:pd.DataFrame
        index:样本名
        columns:样本属性
    length:dict
        key:基因名/特征ID
        value:基因长度
    """
    
    def __init__(
        self,
        counts: pd.DataFrame=None,
        metadata: pd.DataFrame=None,
        length: dict=None
    ):
        ## 构造函数，初始化CountData对象
        ## counts: 原始表达计数矩阵，基因×样本
        ## metadata: 样本元数据，包含样本属性信息
        ## length: 基因长度字典，用于FPKM/TPM计算
        self._validate(counts,metadata,length)
        self._counts=counts.copy() if counts is not None else None
        self._metadata=metadata.copy() if metadata is not None else None
        self._length=length.copy() if length is not None else None

    def _validate(self,counts,metadata,length):
        ## 内部验证函数，检查输入数据的有效性
        ## counts: 检查缺失值、负值、非整数值，以及是否与length/metadata索引匹配
        ## metadata: 检查缺失值
        if counts is not None:
            if counts.isnull().any().any():
                raise ValueError("counts 不能包含缺失值")
            if (counts<0).any().any():
                raise ValueError("counts 不能包含负值")
            if (counts % 1 != 0).any().any():
                raise ValueError("counts 不能包含非整数值")
            if length is not None:
                if not all(gene in counts.index for gene in length.keys()):
                    raise ValueError("counts 索引 与 length 键值不匹配")
            if metadata is not None:
                if not all(sample in metadata.index for sample in counts.columns):
                    raise ValueError("metadata 索引 与 counts 列名不匹配")
        if metadata is not None:
            if metadata.isnull().any().any():
                raise ValueError("metadata 不能包含缺失值")
        


    @classmethod
    def writein(cls,counts_path:str=None,metadata_path:str=None,length_path:str=None):
        ## 类方法：从CSV文件读取数据并创建CountData对象
        ## counts_path: counts矩阵文件路径，第一列需命名为"id"
        ## metadata_path: 元数据文件路径，第一列需命名为"run"
        ## length_path: 基因长度文件路径，第一列"id"，第二列"length"
        ## 返回: CountData实例
        # 标准读取（自动检测逗号分隔）
        counts_df=metadata_df=length_dict=None
        if counts_path is not None:
            if not counts_path.endswith(".csv"):
                raise Exception("仅支持.csv格式")
            counts_df=pd.read_csv(counts_path)
            if counts_df.columns[0].lower() != "id":
                raise ValueError("第一列列名不是ID")
            counts_df.set_index(counts_df.columns[0],inplace=True)

        if metadata_path is not None:
            if not metadata_path.endswith(".csv"):
                raise Exception("仅支持.csv格式")
            metadata_df=pd.read_csv(metadata_path)
            if metadata_df.columns[0].lower() != "run":
                raise ValueError("第一列列名不是run")
            metadata_df.set_index(metadata_df.columns[0],inplace=True)
        
        if length_path is not None:
            if not length_path.endswith(".csv"):
                raise Exception("仅支持.csv格式")
            length_df=pd.read_csv(length_path)
            length_df.columns = length_df.columns.str.lower()
            if length_df.columns[0] != "id":
                raise ValueError("第一列列名不是ID")
            if length_df.columns[1] != "length":
                raise ValueError("第二列列名不是length")
            length_df.set_index(length_df.columns[0],inplace=True)
            length_dict=length_df['length'].to_dict()
        
        return cls(counts=counts_df, metadata=metadata_df,length=length_dict)


    @property
    def counts(self) -> pd.DataFrame:
        ## 属性：返回counts矩阵的副本
        return self._counts.copy()
    
    @property
    def metadata(self) -> pd.DataFrame:
        ## 属性：返回metadata的副本
        return self._metadata.copy()
    
    @property
    def length(self) -> dict:
        ## 属性：返回基因长度字典的副本
        return self._length.copy()
    
    @property
    def shape(self) -> tuple:
        ## 属性：返回counts矩阵的维度 (基因数, 样本数)
        return self._counts.shape
    
    @property
    def n_genes(self) -> int:
        ## 属性：返回基因数量
        return self._counts.shape[0]
    
    @property
    def n_samples(self) -> int:
        ## 属性：返回样本数量
        return self._counts.shape[1]
    
    @property
    def gene_names(self) -> list:
        ## 属性：返回所有基因名称列表
        return self._counts.index.tolist()
    
    @property
    def sample_names(self) -> list:
        ## 属性：返回所有样本名称列表
        return self._counts.columns.tolist()

    @property
    def sample_reads_sum(self) -> dict:
        ## 属性：计算每个样本的总reads数
        ## 返回: {样本名: 总reads数}
        sum_dict={}
        for sample in self.sample_names:
            sum_dict[sample]=self._counts[sample].sum()
        return sum_dict

    @property
    def calculate_CPM(self) -> pd.DataFrame:
        ## 属性：计算CPM (Counts Per Million) 归一化
        ## CPM = (counts / 样本总reads数) * 1,000,000
        ## 返回: CPM矩阵，基因×样本
        sample_reads_sum=pd.Series(self.sample_reads_sum)
        calculate_CPM=self._counts/sample_reads_sum*1000000
        calculate_CPM.index=self._counts.index
        return calculate_CPM
            
    @property
    def calculate_FPK(self) -> pd.DataFrame:
        ## 属性：计算FPK (Fragments Per Kilobase) - FPKM的中间步骤
        ## FPK = (counts / 基因长度) * 1000
        ## 需要length字典支持
        length_series=pd.Series(self._length)
        common_genes=self._counts.index.intersection(length_series.index)
        counts_aligned=self._counts.loc[common_genes]
        length_aligned=length_series[common_genes]
        calculate_FPK=pd.DataFrame()
        for sample in counts_aligned.columns:
            calculate_FPK[sample]=counts_aligned[sample]/length_aligned*1000
        calculate_FPK.index=counts_aligned.index
        calculate_FPK.index.name=self._counts.index.name
        return calculate_FPK

    @property
    def calculate_FPKM(self) -> pd.DataFrame:
        ## 属性：计算FPKM (Fragments Per Kilobase per Million)
        ## FPKM = FPK / (样本总reads数) * 1,000,000
        ## 返回: FPKM矩阵，基因×样本
        fpk=self.calculate_FPK
        sample_reads_sum=self.sample_reads_sum
        calculate_FPKM=pd.DataFrame()
        for sample in fpk.columns:
            calculate_FPKM[sample]=fpk[sample]/sample_reads_sum[sample]*1000000
        calculate_FPKM.index=fpk.index
        calculate_FPKM.index.name=self._counts.index.name
        return calculate_FPKM

    @property
    def calculate_TPM(self) -> pd.DataFrame:
        ## 属性：计算TPM (Transcripts Per Million)
        ## TPM = (FPK / 样本总FPK) * 1,000,000
        ## 需要length字典支持
        ## 返回: TPM矩阵，基因×样本
        if self._length==None:
            raise ValueError("计算TPM需要length信息")
        fpk=self.calculate_FPK
        sample_sums=fpk.sum(axis=0)
        calculate_TPM=pd.DataFrame()
        for sample in fpk.columns:
            calculate_TPM[sample]=fpk[sample]/sample_sums[sample]*1000000
        calculate_TPM.index=fpk.index
        calculate_TPM.index.name=self._counts.index.name
        return calculate_TPM

    def filter_low_expression(self,method:str="TPM",filter_threshold:float=0) -> pd.DataFrame:
        ## 基于归一化表达值过滤低表达基因
        ## method: 归一化方法，可选"CPM"/"TPM"/"FPKM"
        ## filter_threshold: 表达阈值，保留至少在一个样本中表达量>该阈值的基因
        ## 返回: 过滤后的表达矩阵
        ## 打印: 保留基因比例
        if method=="CPM":
            to_be_filtered=self.calculate_CPM
        elif method=="TPM":
            to_be_filtered=self.calculate_TPM
        elif method=="FPKM":
            to_be_filtered=self.calculate_FPKM
        else:
            raise ValueError("归一化方法必须选择CPM/TPM/FPKM")
        filtered=to_be_filtered[(to_be_filtered>filter_threshold).any(axis=1)]
        kept_gene_ratio=filtered.shape[0]/to_be_filtered.shape[0]
        print(f"kept_gene_ratio={kept_gene_ratio}")
        return filtered

    def pca(self,method:str="TPM", n_components:int=2,filter_threshold:float=0,label_name:str=None):
        ## 执行主成分分析(PCA)
        ## method: 归一化方法，用于过滤低表达基因
        ## n_components: PCA主成分数量
        ## filter_threshold: 低表达过滤阈值
        ## label_name: metadata中的列名，用于将样本标签替换为分组标签
        ## 返回: (pca_df, pca_var_ratio) - PCA结果DataFrame和各主成分解释方差比例
        if method=="TPM":
            expr=self.filter_low_expression(method="TPM",filter_threshold=filter_threshold)
        elif method=="FPKM":
            expr=self.filter_low_expression(method="FPKM",filter_threshold=filter_threshold)
        elif method=="CPM":
            expr=self.filter_low_expression(method="CPM",filter_threshold=filter_threshold)
        else:
            raise ValueError("归一化方法必须选择CPM/TPM/FPKM")
        expr=expr.T
        if self._metadata is not None and label_name is not None:
            expr.index=self._metadata[label_name]
        pca=PCA(n_components=2)
        pca_result=pca.fit_transform(expr)
        pca_df=pd.DataFrame(pca_result,columns=[f'PC{i+1}'for i in range(n_components)], index=expr.index)
        pca_var_ratio=pca.explained_variance_ratio_
        return pca_df, pca_var_ratio
    
    def plot_pca(self,method:str="TPM",filter_threshold:float=0,label_name:str=None,figsize:tuple=(10, 8),show_labels=True):
        ## 绘制PCA散点图
        ## method: 归一化方法
        ## filter_threshold: 过滤阈值
        ## label_name: 样本标签列名
        ## figsize: 图形尺寸
        ## show_labels: 是否显示样本标签
        ## 返回: matplotlib Figure对象
        pca_df, pca_var_ratio = self.pca(method=method, n_components=2, filter_threshold=filter_threshold,label_name=label_name)
        fig, ax = plt.subplots(figsize=figsize)
        ax.scatter(pca_df['PC1'], pca_df['PC2'], s=50, alpha=0.7, color='steelblue')
        ax.set_xlabel(f'PC1 ({pca_var_ratio[0]:.2%} variance)', fontsize=12)
        ax.set_ylabel(f'PC2 ({pca_var_ratio[1]:.2%} variance)', fontsize=12)
        ax.set_title(f'PCA Plot ({method}, threshold={filter_threshold})', fontsize=14, fontweight='bold')
        if show_labels:
            for sample in pca_df.index:
                ax.annotate(sample, (pca_df.loc[sample, 'PC1'], pca_df.loc[sample, 'PC2']), xytext=(5, 5), textcoords='offset points', fontsize=8)
        ax.grid(True,alpha=0.3)
        plt.tight_layout()
        return fig
    
    def sample_correlation(self,method:str="TPM",filter_threshold:float=0,label_name:str=None):
        ## 计算样本间Pearson相关系数矩阵
        ## method: 归一化方法
        ## filter_threshold: 过滤阈值
        ## label_name: 将样本名替换为metadata中的分组标签
        ## 返回: 相关系数矩阵DataFrame
        if method=="TPM":
            expr=self.filter_low_expression(method="TPM",filter_threshold=filter_threshold)
        elif method=="FPKM":
            expr=self.filter_low_expression(method="FPKM",filter_threshold=filter_threshold)
        elif method=="CPM":
            expr=self.filter_low_expression(method="CPM",filter_threshold=filter_threshold)
        else:
            raise ValueError("归一化方法必须选择CPM/TPM/FPKM")
        corr_matrix=expr.corr(method='pearson')
        if self._metadata is not None and label_name is not None:
            corr_matrix.columns=self._metadata[label_name]
            corr_matrix.index=self._metadata[label_name]
        return corr_matrix

    def plot_sample_correlation(self,method:str="TPM",filter_threshold:float=0,label_name:str=None,cmap:str='coolwarm'):
        ## 绘制样本相关性热图
        ## method: 归一化方法
        ## filter_threshold: 过滤阈值
        ## label_name: 样本标签列名
        ## cmap: 颜色映射
        ## 返回: matplotlib Figure对象
        corr_matrix=self.sample_correlation(method=method,filter_threshold=filter_threshold,label_name=label_name)
        fig,ax=plt.subplots(figsize=(10,8))
        im=ax.imshow(corr_matrix, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
        plt.colorbar(im, ax=ax, label='Pearson Correlation')
        ax.set_xticks(np.arange(len(corr_matrix.columns)))
        ax.set_yticks(np.arange(len(corr_matrix.index)))
        ax.set_xticklabels(corr_matrix.columns,rotation=45,ha='right',fontsize=10)
        ax.set_yticklabels(corr_matrix.columns,fontsize=10)
        for i in range(len(corr_matrix.index)):
            for j in range(len(corr_matrix.columns)):
                text=ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',ha="center",va="center",color="black",fontsize=8)
        ax.set_title(f'Sample Correlation Heatmap ({method}, threshold={filter_threshold})', fontsize=14, fontweight='bold')
        ax.set_xlabel('Samples',fontsize=12)
        ax.set_ylabel('Samples',fontsize=12)
        plt.tight_layout()
        return fig

    def group_samples(self,grouped_by:str=None,groups:list=None) -> dict:
        ## 根据元数据将样本分组
        ## grouped_by: metadata中的列名，按该列的唯一值分组
        ## groups: 手动指定的组名列表，从metadata中筛选包含这些组名的样本
        ## 必须且只能选择grouped_by或groups两种方法其一
        ## 返回: {组名: [样本列表]}
        group_dict={}
        if grouped_by is not None and groups is None:
            group_names=self._metadata[grouped_by].unique()
            for group_name in group_names:
                samples=[]
                mask=(self._metadata[grouped_by]==group_name)
                samples=self._metadata[mask].index.tolist()
                group_dict[group_name]=samples
        elif grouped_by is None and groups is not None:
            group_names=groups
            for group_name in group_names:
                samples=[]
                mask=(self._metadata==group_name).any(axis=1)
                samples=self._metadata[mask].index.tolist()
                group_dict[group_name]=samples
        else:
            raise ValueError("必须指定grouped_by或groups两者其一")
        return group_dict
   

    def differential_test(self,method:str="TPM",filter_threshold:float=0,grouped_by:str=None,groups:list=None,log2_transform:bool=True,fdr_correction:bool=True,test_method:str="t_test",paired:bool=False):
        ## 执行差异表达分析（两组比较）
        ## method: 归一化方法
        ## filter_threshold: 过滤低表达基因的阈值
        ## grouped_by: metadata列名，自动确定两组
        ## groups: 手动指定两组名称的列表，如["control","treatment"]
        ## log2_transform: 是否对表达值进行log2(x+1)变换
        ## fdr_correction: 是否进行FDR多重检验校正
        ## test_method: 统计检验方法，"t_test"或"wilcoxon"
        ## paired: 是否配对检验
        ## 返回: DataFrame包含统计量、log2FC、p值、FDR校正p值、显著性标记
        if method=="TPM":
            expr=self.filter_low_expression(method="TPM",filter_threshold=filter_threshold)
        elif method=="FPKM":
            expr=self.filter_low_expression(method="FPKM",filter_threshold=filter_threshold)
        elif method=="CPM":
            expr=self.filter_low_expression(method="CPM",filter_threshold=filter_threshold)
        else:
            raise ValueError("归一化方法必须选择CPM/TPM/FPKM")

        if log2_transform:
            expr=np.log2(expr+1)

        group_dict=self.group_samples(grouped_by=grouped_by,groups=groups)
        if len(group_dict.keys())!=2:
            raise Exception("t-test要求分组数为2")
        for key in group_dict.keys():
            if len(group_dict[key])<2:
                raise Exception("每组需要至少两个样本")
        
        group1_samples=group_dict[list(group_dict.keys())[0]]
        group2_samples=group_dict[list(group_dict.keys())[1]]
        group1_name=list(group_dict.keys())[0]
        group2_name=list(group_dict.keys())[1]

        statistics=[]
        p_values=[]
        log2_fold_changes=[]

        for idx, row in expr.iterrows():
            group1_vals=row[group1_samples].values
            group2_vals=row[group2_samples].values
            if log2_transform:
                lfc=np.mean(group2_vals)-np.mean(group1_vals)
            else:
                lfc=np.log2((np.mean(group2_vals)+1)/(np.mean(group1_vals)+1))
            log2_fold_changes.append(lfc)
            if test_method=="t_test":
                if paired==True:
                    stat,p_val=stats.ttest_rel(group1_vals,group2_vals,nan_policy='omit')
                else:
                    stat,p_val=stats.ttest_ind(group1_vals,group2_vals,nan_policy='omit')
            if test_method=="wilcoxon":
                if paired==True:
                    stat,p_val=stats.wilcoxon(group1_vals,group2_vals,nan_policy='omit')
                else:
                    stat,p_val=stats.mannwhitneyu(group1_vals,group2_vals,alternative='two-sided',nan_policy='omit')
            statistics.append(stat)
            p_values.append(p_val)
        
        result_df = pd.DataFrame({f"{test_method}":statistics,"log2_fold_changes":log2_fold_changes,"p_values":p_values},index=expr.index)
        if fdr_correction:
            result_df["fdr"]=false_discovery_control(result_df["p_values"],method='bh')
            result_df['significance']=self.significance_symbol(result_df["fdr"])
        else:
            result_df['significance']=self.significance_symbol(result_df["p_values"])
        return result_df

    @staticmethod
    def significance_symbol(values:pd.Series) -> list:
        ## 静态方法：根据p值/FDR值返回显著性标记符号
        ## values: p值或FDR值序列
        ## 返回: 符号列表 - 'ns'(p>0.05), '*'(p≤0.05), '**'(p≤0.01), '***'(p≤0.001), '****'(p≤0.0001)
        significance=[]
        for value in values:
            if value>0.05:
                significance.append('ns')
            elif value>0.01:
                significance.append('*')
            elif value>0.001:
                significance.append('**')
            elif value>0.0001:
                significance.append('***')
            else:
                significance.append('****')
        return significance
    
    def pick_significance(self,method:str="TPM",filter_threshold:float=0,grouped_by:str=None,groups:list=None,log2_transform:bool=True,fdr_correction:bool=True,test_method:str="t_test",paired:bool=False,significance_threshold:float=0.05):
        ## 筛选显著差异表达基因
        ## 参数同differential_test，额外参数:
        ## significance_threshold: 显著性阈值（p值或FDR）
        ## log2fc_threshold: log2 fold change阈值（本函数中定义但未使用该阈值进行筛选，仅筛选p值）
        ## 返回: 仅包含显著差异表达基因的DataFrame
        result_df=self.differential_test(method=method,filter_threshold=filter_threshold,grouped_by=grouped_by,groups=groups,log2_transform=log2_transform,fdr_correction=fdr_correction,test_method=test_method,paired=paired)
        
        if fdr_correction:
            p_col = "fdr"
        else:
            p_col = "p_values"
        significance_df=result_df[result_df[p_col]<significance_threshold]
        return significance_df
        
    def plot_differential_test(self,method:str="TPM",filter_threshold:float=0,grouped_by:str=None,groups:list=None,log2_transform:bool=True,fdr_correction:bool=True,test_method:str="t_test",paired:bool=False,significance_threshold:float=0.05,log2fc_threshold:float=2):
        ## 绘制火山图(Volcano Plot)展示差异表达结果
        ## significance_threshold: 显著性阈值
        ## log2fc_threshold: log2 fold change阈值，用于定义上调/下调基因
        ## 返回: matplotlib Figure对象
        result_df=self.differential_test(method=method,filter_threshold=filter_threshold,grouped_by=grouped_by,groups=groups,log2_transform=log2_transform,fdr_correction=fdr_correction,test_method=test_method,paired=paired)
        fig,ax=plt.subplots(figsize=(10, 8))
        
        if fdr_correction:
            p_col = "fdr"
        else:
            p_col = "p_values"
        
        log2fc = result_df["log2_fold_changes"]
        p_vals = result_df[p_col]
        
        significant = p_vals < significance_threshold
        up_regulated = significant & (log2fc > log2fc_threshold)
        down_regulated = significant & (log2fc < -log2fc_threshold)
        not_significant = ~significant | ((log2fc >= -log2fc_threshold) & (log2fc <= log2fc_threshold))
        
        ax.scatter(log2fc[not_significant], -np.log10(p_vals[not_significant]), 
                alpha=0.5, s=10, c="grey", label='Not significant')
        ax.scatter(log2fc[up_regulated], -np.log10(p_vals[up_regulated]), 
                alpha=0.7, s=15, c="red", label=f'Up-regulated (log2FC > {log2fc_threshold})')
        ax.scatter(log2fc[down_regulated], -np.log10(p_vals[down_regulated]), 
                alpha=0.7, s=15, c="blue", label=f'Down-regulated (log2FC < -{log2fc_threshold})')
        
        ax.axhline(y=-np.log10(significance_threshold), color='red', linestyle='--', alpha=0.5)
        ax.axvline(x=log2fc_threshold, color='red', linestyle='--', alpha=0.5)
        ax.axvline(x=-log2fc_threshold, color='red', linestyle='--', alpha=0.5)
    
        ax.set_xlabel('log2(Fold Change)', fontsize=12, fontweight='bold')
        ax.set_ylabel('-log10(p-value)', fontsize=12, fontweight='bold')
        
        group_dict=self.group_samples(grouped_by=grouped_by,groups=groups)
        groups=list(group_dict.keys())
        test_name = "Paired " + test_method if paired else test_method
        correction_text = " (FDR corrected)" if fdr_correction else ""
        title = f'Volcano Plot - {test_name}{correction_text}\nGroups: {groups[0]} vs {groups[1]}'
        ax.set_title(title, fontsize=14, fontweight='bold') 
        ax.legend(loc='best', frameon=True, fancybox=True, shadow=True)

        plt.tight_layout()
        return fig

    @classmethod
    def writeout(cls,file,file_name:str):
        ## 类方法：将DataFrame或matplotlib Figure保存为文件
        ## file: pd.DataFrame或plt.Figure对象
        ## file_name: 输出文件名（不含扩展名）
        ## DataFrame保存为CSV，Figure保存为PNG
        if isinstance(file, pd.DataFrame):
            csv_file=f"{file_name}.csv"
            file.to_csv(csv_file, index=False, encoding='utf-8-sig')
            print(f"DataFrame已保存为: {csv_file}")
        if isinstance(file, plt.Figure):
            img_file=f"{file_name}.png"
            file.savefig(img_file, dpi=300, bbox_inches='tight')
            print(f"图片已保存为: {img_file}")

In [18]:
counts, metadata, length = generate_test_data_fixed(seed=12,n_groups=2)
test = CountData(counts=counts, metadata=metadata, length=length)
plot=test.differential_test(filter_threshold=100,grouped_by="condition")
print(plot)
CountData.writeout(plot,file_name="differential_test")


kept_gene_ratio=0.728
             t_test  log2_fold_changes  p_values       fdr significance
gene_1     3.484579          -0.720121  0.008264  0.017189            *
gene_2    -0.033792           0.045471  0.973871  0.991577           ns
gene_3     3.365853          -0.631944  0.009846  0.019692            *
gene_4     0.963911          -1.155885  0.363319  0.479160           ns
gene_6     3.900911          -0.747468  0.004539  0.012672            *
...             ...                ...       ...       ...          ...
gene_995   3.231321          -0.675522  0.012033  0.022932            *
gene_996   2.506823          -2.039819  0.036551  0.060202           ns
gene_998   4.326163          -0.774809  0.002525  0.009182           **
gene_999   0.211838          -0.228829  0.837530  0.900624           ns
gene_1000  3.703708          -0.800219  0.006010  0.014472            *

[728 rows x 5 columns]
DataFrame已保存为: differential_test.csv
